# Mol2Vec Encoder


In [4]:
# =========================================
# GRU on Mol2Vec
# Unsupervised 


import random
import numpy as np
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


train_df = pd.read_csv("R1D100_train.csv")
test_df = pd.read_csv("R1D100_test.csv")


X_train = train_df.iloc[:, 1:-1].values.astype(np.float32)
X_test  = test_df.iloc[:, 1:-1].values.astype(np.float32)

y_train = train_df.iloc[:, -1].values
y_test  = test_df.iloc[:, -1].values


class Mol2VecDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        return torch.tensor(x, dtype=torch.float32)

train_dataset = Mol2VecDataset(X_train)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
    worker_init_fn=lambda worker_id: np.random.seed(SEED)
)

# =========================================
# GRU model
# =========================================
class Mol2VecBiGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim=32):
        super().__init__()

        self.bigru = nn.GRU(
            input_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=False
        )

    def forward(self, x):
        # x: (B, D) -> (B, seq_len=1, D)
        x = x.unsqueeze(1)

        out, _ = self.bigru(x)

        # out: (B, seq_len=1, hidden_dim)
        pooled = out.squeeze(1)

        return pooled

device = torch.device("cpu")

model = Mol2VecBiGRU(
    input_dim=X_train.shape[1]
).to(device)

# =========================================
#  Optimizer
# =========================================
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# =========================================
# Training (Unsupervised ปกติ)
# =========================================
EPOCHS = 100

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x in train_loader:

        x = x.to(device)

        optimizer.zero_grad()

        z = model(x)

        # dummy unsupervised objective
        loss = torch.mean(z ** 2)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS} Loss: {total_loss/len(train_loader):.4f}")

# =========================================
# Generate embeddings
# =========================================
model.eval()

with torch.no_grad():

    train_emb = model(
        torch.tensor(X_train, dtype=torch.float32).to(device)
    ).cpu().numpy()

    test_emb = model(
        torch.tensor(X_test, dtype=torch.float32).to(device)
    ).cpu().numpy()

# =========================================
# 7. Save embeddings
# =========================================
pd.DataFrame(train_emb).to_csv(
    "mol2vecGRU-R1D100_train.csv",
    index=False
)

pd.DataFrame(test_emb).to_csv(
    "mol2vecGRU-R1D100_test.csv",
    index=False
)

print("Embeddings saved.")

Epoch 1/100 Loss: 0.0018
Epoch 2/100 Loss: 0.0002
Epoch 3/100 Loss: 0.0001
Epoch 4/100 Loss: 0.0001
Epoch 5/100 Loss: 0.0001
Epoch 6/100 Loss: 0.0000
Epoch 7/100 Loss: 0.0000
Epoch 8/100 Loss: 0.0000
Epoch 9/100 Loss: 0.0000
Epoch 10/100 Loss: 0.0000
Epoch 11/100 Loss: 0.0000
Epoch 12/100 Loss: 0.0000
Epoch 13/100 Loss: 0.0000
Epoch 14/100 Loss: 0.0000
Epoch 15/100 Loss: 0.0000
Epoch 16/100 Loss: 0.0000
Epoch 17/100 Loss: 0.0000
Epoch 18/100 Loss: 0.0000
Epoch 19/100 Loss: 0.0000
Epoch 20/100 Loss: 0.0000
Epoch 21/100 Loss: 0.0000
Epoch 22/100 Loss: 0.0000
Epoch 23/100 Loss: 0.0000
Epoch 24/100 Loss: 0.0000
Epoch 25/100 Loss: 0.0000
Epoch 26/100 Loss: 0.0000
Epoch 27/100 Loss: 0.0000
Epoch 28/100 Loss: 0.0000
Epoch 29/100 Loss: 0.0000
Epoch 30/100 Loss: 0.0000
Epoch 31/100 Loss: 0.0000
Epoch 32/100 Loss: 0.0000
Epoch 33/100 Loss: 0.0000
Epoch 34/100 Loss: 0.0000
Epoch 35/100 Loss: 0.0000
Epoch 36/100 Loss: 0.0000
Epoch 37/100 Loss: 0.0000
Epoch 38/100 Loss: 0.0000
Epoch 39/100 Loss: 0.